# 🚗 AutoWorth AI — Used Car Price & Deal Advisor

## Final Machine Learning Project

**Problem:** Regression  
**Target:** `price`

The idea of the project is to use used-car data from different manufacturers, predict the expected market price of a car, then compare it with the seller's asking price.

### Manufacturers used

- Audi
- BMW
- Ford
- Hyundai
- Mercedes
- Skoda
- Toyota
- Vauxhall
- Volkswagen

The separate `focus.csv` and `cclass.csv` files are not used because Focus already exists inside the Ford dataset and C-Class already exists inside the Mercedes dataset. Using them again would repeat some data.

The `unclean focus.csv` and `unclean cclass.csv` files are also not part of the main project dataset.

## 1. Import Libraries

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import joblib

%matplotlib inline
sns.set_style("whitegrid")

## 2. Load the Datasets

Upload the 9 manufacturer CSV files to Colab first.

Hyundai has a small difference in the tax column name (`tax(£)`), so I rename it to `tax` before combining the datasets.

In [ ]:
audi = pd.read_csv("data/audi.csv")
bmw = pd.read_csv("data/bmw.csv")
ford = pd.read_csv("data/ford.csv")
hyundai = pd.read_csv("data/hyundi.csv")
mercedes = pd.read_csv("data/merc.csv")
skoda = pd.read_csv("data/skoda.csv")
toyota = pd.read_csv("data/toyota.csv")
vauxhall = pd.read_csv("data/vauxhall.csv")
volkswagen = pd.read_csv("data/vw.csv")

FileNotFoundError: [Errno 2] No such file or directory: '/content/audi.csv'

In [ ]:
print("Audi:", audi.shape)
print("BMW:", bmw.shape)
print("Ford:", ford.shape)
print("Hyundai:", hyundai.shape)
print("Mercedes:", mercedes.shape)
print("Skoda:", skoda.shape)
print("Toyota:", toyota.shape)
print("Vauxhall:", vauxhall.shape)
print("Volkswagen:", volkswagen.shape)

### Add a Make column

In [ ]:
audi["Make"] = "Audi"
bmw["Make"] = "BMW"
ford["Make"] = "Ford"
hyundai["Make"] = "Hyundai"
mercedes["Make"] = "Mercedes"
skoda["Make"] = "Skoda"
toyota["Make"] = "Toyota"
vauxhall["Make"] = "Vauxhall"
volkswagen["Make"] = "Volkswagen"

hyundai = hyundai.rename(columns={"tax(£)": "tax"})

## 3. Inspect the Data Before Combining

In [ ]:
audi.head()

In [ ]:
audi.info()

In [ ]:
audi.describe()

In [ ]:
print("Audi missing values:")
print(audi.isnull().sum())

print("\nAudi duplicates:", audi.duplicated().sum())

print("\nAudi transmission values:")
print(audi["transmission"].value_counts())

print("\nAudi fuel types:")
print(audi["fuelType"].value_counts())

The same columns exist in the manufacturer files after fixing Hyundai's tax column, so the datasets can now be combined.

## 4. Combine All Manufacturers

In [ ]:
df = pd.concat(
    [audi, bmw, ford, hyundai, mercedes, skoda, toyota, vauxhall, volkswagen],
    ignore_index=True
)

df.head()

In [ ]:
print("Combined dataset shape:", df.shape)
print("\nColumns:")
print(df.columns)

In [ ]:
df.info()

In [ ]:
df.describe(include="all").T

## 5. Data Quality Checks

In [ ]:
print("Missing values:")
print(df.isnull().sum())

print("\nTotal duplicate rows:")
print(df.duplicated().sum())

In [ ]:
print("Year range:", df["year"].min(), "to", df["year"].max())
print("Price range:", df["price"].min(), "to", df["price"].max())
print("Mileage range:", df["mileage"].min(), "to", df["mileage"].max())
print("Engine size range:", df["engineSize"].min(), "to", df["engineSize"].max())
print("MPG range:", df["mpg"].min(), "to", df["mpg"].max())
print("Tax range:", df["tax"].min(), "to", df["tax"].max())

In [ ]:
print("Transmission values:")
print(df["transmission"].value_counts())

print("\nFuel type values:")
print(df["fuelType"].value_counts())

print("\nManufacturers:")
print(df["Make"].value_counts())

### What I found

- The combined dataset has no missing values after fixing Hyundai's tax column.
- There are exact duplicate rows.
- There is one impossible year value: `2060`.
- Some values look unusual, such as very high MPG or engine size equal to 0, but they should be investigated before deleting them.

For example, some BMW i3 records have unusual MPG values and zero engine size because they are electric/hybrid vehicles. They should not automatically be treated as errors.

In [ ]:
df[df["year"] > 2021]

In [ ]:
df[df["engineSize"] == 0][
    ["Make", "model", "year", "fuelType", "engineSize", "mpg", "price"]
].head(20)

In [ ]:
df.sort_values("mpg", ascending=False)[
    ["Make", "model", "year", "fuelType", "mpg", "engineSize", "price"]
].head(15)

## 6. Data Cleaning

### Remove exact duplicates

In [ ]:
print("Shape before removing duplicates:", df.shape)

df = df.drop_duplicates().reset_index(drop=True)

print("Shape after removing duplicates:", df.shape)
print("Duplicates left:", df.duplicated().sum())

### Clean text columns

In [ ]:
df["model"] = df["model"].str.strip()
df["transmission"] = df["transmission"].str.strip()
df["fuelType"] = df["fuelType"].str.strip()
df["Make"] = df["Make"].str.strip()

### Remove the impossible year

In [ ]:
print(df[df["year"] > 2021])

df = df[df["year"] <= 2021].copy()
df = df.reset_index(drop=True)

print("Maximum year after cleaning:", df["year"].max())

I keep the other unusual values unless there is clear evidence that they are wrong. An unusual value is not automatically an error.

## 7. Outlier Investigation

In [ ]:
plt.figure(figsize=(15, 4))

plt.subplot(1, 3, 1)
sns.boxplot(x=df["price"])
plt.title("Price")

plt.subplot(1, 3, 2)
sns.boxplot(x=df["mileage"])
plt.title("Mileage")

plt.subplot(1, 3, 3)
sns.boxplot(x=df["engineSize"])
plt.title("Engine Size")

plt.tight_layout()
plt.show()

In [ ]:
Q1 = df["price"].quantile(0.25)
Q3 = df["price"].quantile(0.75)
IQR = Q3 - Q1

lower_price = Q1 - 1.5 * IQR
upper_price = Q3 + 1.5 * IQR

print("Price lower IQR limit:", lower_price)
print("Price upper IQR limit:", upper_price)
print("Number of prices outside the IQR limits:",
      ((df["price"] < lower_price) | (df["price"] > upper_price)).sum())

Cars outside the IQR limits are not deleted automatically. Expensive cars can still be valid real cars, especially because the dataset contains several premium manufacturers.

## 8. Exploratory Data Analysis

### 1. Price Distribution

In [ ]:
plt.figure(figsize=(9, 5))
sns.histplot(df["price"], bins=50, kde=True)
plt.title("Price Distribution")
plt.xlabel("Price (£)")
plt.show()

### 2. Price vs Mileage

In [ ]:
sample = df.sample(15000, random_state=42)

plt.figure(figsize=(9, 5))
sns.scatterplot(data=sample, x="mileage", y="price", alpha=0.4)
plt.title("Price vs Mileage")
plt.xlabel("Mileage")
plt.ylabel("Price (£)")
plt.show()

print("Correlation:", df["mileage"].corr(df["price"]))

### 3. Price vs Year

In [ ]:
year_prices = df.groupby("year")["price"].median().reset_index()

plt.figure(figsize=(11, 5))
sns.lineplot(data=year_prices, x="year", y="price", marker="o")
plt.title("Median Price by Year")
plt.ylabel("Median Price (£)")
plt.show()

### 4. Average Price by Manufacturer

In [ ]:
make_prices = df.groupby("Make")["price"].mean().sort_values(ascending=False)

print(make_prices)

plt.figure(figsize=(10, 5))
sns.barplot(x=make_prices.values, y=make_prices.index)
plt.title("Average Price by Manufacturer")
plt.xlabel("Average Price (£)")
plt.show()

### 5. Price by Transmission

In [ ]:
plt.figure(figsize=(9, 5))
sns.boxplot(data=df, x="transmission", y="price", showfliers=False)
plt.title("Price by Transmission")
plt.xticks(rotation=20)
plt.show()

print(df.groupby("transmission")["price"].median().sort_values(ascending=False))

### 6. Price by Fuel Type

In [ ]:
plt.figure(figsize=(9, 5))
sns.boxplot(data=df, x="fuelType", y="price", showfliers=False)
plt.title("Price by Fuel Type")
plt.xticks(rotation=20)
plt.show()

print(df.groupby("fuelType")["price"].median().sort_values(ascending=False))

### 7. Correlation Heatmap

In [ ]:
numeric_columns = ["year", "price", "mileage", "tax", "mpg", "engineSize"]

plt.figure(figsize=(8, 6))
sns.heatmap(df[numeric_columns].corr(), annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Correlation Heatmap")
plt.show()

## 9. Separate Features and Target

`price` is the target, so it must not be included in `X`.

The seller's asking price is also not part of the training data. It will only be used later in the Deal Advisor.

In [ ]:
X = df.drop("price", axis=1)
y = df["price"]

print("X shape:", X.shape)
print("y shape:", y.shape)

## 10. Train / Validation / Test Split

The split is:

- 70% training
- 15% validation
- 15% test

The test set will not be used while comparing or tuning models.

In [ ]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=42
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    random_state=42
)

print("Training:", X_train.shape)
print("Validation:", X_val.shape)
print("Test:", X_test.shape)

## 11. Feature Engineering

I created three features.

### CarAge
The newest cars in the dataset are from 2020, so I use 2021 as a fixed reference year.

### MileagePerYear
This represents how heavily the car has been used for its age.

### CarAgeSquared
Car depreciation is not always perfectly linear. Squaring age gives the model another way to represent the stronger effect of ageing on older cars.

In [ ]:
X_train["CarAge"] = 2021 - X_train["year"]
X_val["CarAge"] = 2021 - X_val["year"]
X_test["CarAge"] = 2021 - X_test["year"]

In [ ]:
X_train["MileagePerYear"] = X_train["mileage"] / X_train["CarAge"].replace(0, 1)
X_val["MileagePerYear"] = X_val["mileage"] / X_val["CarAge"].replace(0, 1)
X_test["MileagePerYear"] = X_test["mileage"] / X_test["CarAge"].replace(0, 1)

In [ ]:
X_train["CarAgeSquared"] = X_train["CarAge"] ** 2
X_val["CarAgeSquared"] = X_val["CarAge"] ** 2
X_test["CarAgeSquared"] = X_test["CarAge"] ** 2

In [ ]:
X_train[
    ["year", "mileage", "CarAge", "MileagePerYear", "CarAgeSquared"]
].head()

## 12. Numerical and Categorical Features

In [ ]:
numerical_features = [
    "year",
    "mileage",
    "tax",
    "mpg",
    "engineSize",
    "CarAge",
    "MileagePerYear",
    "CarAgeSquared"
]

categorical_features = [
    "Make",
    "model",
    "transmission",
    "fuelType"
]

print("Numerical features:")
print(numerical_features)

print("\nCategorical features:")
print(categorical_features)

## 13. Preprocessing

- Numerical features are standardized.
- Categorical features are One-Hot Encoded.
- The preprocessor is fitted using the **training data only**.
- Validation and test data only use `.transform()`.

This prevents data leakage.

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numerical_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)
    ]
)

X_train_processed = preprocessor.fit_transform(X_train)
X_val_processed = preprocessor.transform(X_val)
X_test_processed = preprocessor.transform(X_test)

print("Training data after preprocessing:", X_train_processed.shape)
print("Validation data after preprocessing:", X_val_processed.shape)
print("Test data after preprocessing:", X_test_processed.shape)

## 14. Model 1 — Linear Regression Baseline

In [ ]:
linear_model = LinearRegression()

linear_model.fit(X_train_processed, y_train)

linear_train_pred = linear_model.predict(X_train_processed)
linear_val_pred = linear_model.predict(X_val_processed)

In [ ]:
linear_train_r2 = r2_score(y_train, linear_train_pred)

linear_r2 = r2_score(y_val, linear_val_pred)
linear_mae = mean_absolute_error(y_val, linear_val_pred)
linear_rmse = np.sqrt(mean_squared_error(y_val, linear_val_pred))

print("Linear Regression")
print("Training R²:", round(linear_train_r2, 4))
print("Validation R²:", round(linear_r2, 4))
print("Validation MAE: £", round(linear_mae, 2))
print("Validation RMSE: £", round(linear_rmse, 2))

## 15. Model 2 — Decision Tree Regressor

In [ ]:
tree_model = DecisionTreeRegressor(random_state=42)

tree_model.fit(X_train_processed, y_train)

tree_train_pred = tree_model.predict(X_train_processed)
tree_val_pred = tree_model.predict(X_val_processed)

In [ ]:
tree_train_r2 = r2_score(y_train, tree_train_pred)

tree_r2 = r2_score(y_val, tree_val_pred)
tree_mae = mean_absolute_error(y_val, tree_val_pred)
tree_rmse = np.sqrt(mean_squared_error(y_val, tree_val_pred))

print("Decision Tree")
print("Training R²:", round(tree_train_r2, 4))
print("Validation R²:", round(tree_r2, 4))
print("Validation MAE: £", round(tree_mae, 2))
print("Validation RMSE: £", round(tree_rmse, 2))

## 16. Model 3 — Random Forest Regressor

In [ ]:
random_forest = RandomForestRegressor(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

random_forest.fit(X_train_processed, y_train)

rf_train_pred = random_forest.predict(X_train_processed)
rf_val_pred = random_forest.predict(X_val_processed)

In [ ]:
rf_train_r2 = r2_score(y_train, rf_train_pred)

rf_r2 = r2_score(y_val, rf_val_pred)
rf_mae = mean_absolute_error(y_val, rf_val_pred)
rf_rmse = np.sqrt(mean_squared_error(y_val, rf_val_pred))

print("Random Forest")
print("Training R²:", round(rf_train_r2, 4))
print("Validation R²:", round(rf_r2, 4))
print("Validation MAE: £", round(rf_mae, 2))
print("Validation RMSE: £", round(rf_rmse, 2))

## 17. Model 4 — Gradient Boosting Regressor

In [ ]:
gradient_boosting = GradientBoostingRegressor(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=3,
    random_state=42
)

gradient_boosting.fit(X_train_processed, y_train)

gb_train_pred = gradient_boosting.predict(X_train_processed)
gb_val_pred = gradient_boosting.predict(X_val_processed)

In [ ]:
gb_train_r2 = r2_score(y_train, gb_train_pred)

gb_r2 = r2_score(y_val, gb_val_pred)
gb_mae = mean_absolute_error(y_val, gb_val_pred)
gb_rmse = np.sqrt(mean_squared_error(y_val, gb_val_pred))

print("Gradient Boosting")
print("Training R²:", round(gb_train_r2, 4))
print("Validation R²:", round(gb_r2, 4))
print("Validation MAE: £", round(gb_mae, 2))
print("Validation RMSE: £", round(gb_rmse, 2))

## 18. Compare the Models

In [ ]:
results = pd.DataFrame({
    "Model": [
        "Linear Regression",
        "Decision Tree",
        "Random Forest",
        "Gradient Boosting"
    ],
    "Train R2": [
        linear_train_r2,
        tree_train_r2,
        rf_train_r2,
        gb_train_r2
    ],
    "Validation R2": [
        linear_r2,
        tree_r2,
        rf_r2,
        gb_r2
    ],
    "MAE": [
        linear_mae,
        tree_mae,
        rf_mae,
        gb_mae
    ],
    "RMSE": [
        linear_rmse,
        tree_rmse,
        rf_rmse,
        gb_rmse
    ]
})

results["R2 Gap"] = results["Train R2"] - results["Validation R2"]

results = results.sort_values("RMSE").reset_index(drop=True)

results

In [ ]:
plt.figure(figsize=(9, 5))
sns.barplot(data=results, x="Validation R2", y="Model")
plt.title("Validation R² Comparison")
plt.xlim(0, 1)
plt.show()

## 19. Check for Overfitting

The training and validation R² scores should be looked at together.

A very high training score with a noticeably lower validation score can indicate overfitting.

In [ ]:
results[["Model", "Train R2", "Validation R2", "R2 Gap"]]

The Decision Tree will usually have an extremely high training score because a deep tree can almost memorize the training data. Random Forest usually reduces this problem by combining many trees.

## 20. Hyperparameter Tuning

Random Forest is one of the strongest models for this dataset, so I tune it using `RandomizedSearchCV`.

For this step I use a Pipeline so that the preprocessing is fitted again inside each cross-validation fold instead of being learned from all training rows before CV.

In [ ]:
tuning_preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numerical_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)
    ]
)

rf_pipeline = Pipeline([
    ("preprocessing", tuning_preprocessor),
    ("model", RandomForestRegressor(random_state=42, n_jobs=-1))
])

In [ ]:
parameters = {
    "model__n_estimators": [50, 75, 100, 125],
    "model__max_depth": [10, 15, 20, None],
    "model__min_samples_split": [2, 5, 10],
    "model__min_samples_leaf": [1, 2, 4],
    "model__max_features": ["sqrt", 0.7, 1.0]
}

random_search = RandomizedSearchCV(
    estimator=rf_pipeline,
    param_distributions=parameters,
    n_iter=6,
    cv=3,
    scoring="neg_root_mean_squared_error",
    random_state=42,
    n_jobs=-1,
    verbose=1
)

random_search.fit(X_train, y_train)

In [ ]:
print("Best parameters:")
print(random_search.best_params_)

print("\nBest cross-validation RMSE:")
print(round(-random_search.best_score_, 2))

## 21. Before Tuning vs After Tuning

In [ ]:
tuned_rf = random_search.best_estimator_

tuned_train_pred = tuned_rf.predict(X_train)
tuned_val_pred = tuned_rf.predict(X_val)

tuned_train_r2 = r2_score(y_train, tuned_train_pred)
tuned_r2 = r2_score(y_val, tuned_val_pred)
tuned_mae = mean_absolute_error(y_val, tuned_val_pred)
tuned_rmse = np.sqrt(mean_squared_error(y_val, tuned_val_pred))

print("Before tuning")
print("Validation R²:", round(rf_r2, 4))
print("MAE: £", round(rf_mae, 2))
print("RMSE: £", round(rf_rmse, 2))

print("\nAfter tuning")
print("Training R²:", round(tuned_train_r2, 4))
print("Validation R²:", round(tuned_r2, 4))
print("MAE: £", round(tuned_mae, 2))
print("RMSE: £", round(tuned_rmse, 2))

## 22. Select the Final Model

I compare the tuned and untuned Random Forest on validation RMSE.

The test set is still untouched.

In [ ]:
if tuned_rmse <= rf_rmse:
    final_model = tuned_rf
    final_model_name = "Tuned Random Forest"
else:
    final_model = Pipeline([
        ("preprocessing", preprocessor),
        ("model", random_forest)
    ])

    final_model.steps[0] = ("preprocessing", preprocessor)
    final_model.steps[1] = ("model", random_forest)

    final_model_name = "Random Forest"

print("Selected final model:", final_model_name)

The cell above keeps the tuned model when it improves validation RMSE.

If the untuned Random Forest is better, the simpler untuned version should be preferred instead of assuming that tuning must improve the model.

## 23. Final Test Evaluation

This is the first time the test target is used to judge the final model.

In [ ]:
if final_model_name == "Tuned Random Forest":
    test_predictions = final_model.predict(X_test)
else:
    test_predictions = random_forest.predict(X_test_processed)

test_r2 = r2_score(y_test, test_predictions)
test_mae = mean_absolute_error(y_test, test_predictions)
test_rmse = np.sqrt(mean_squared_error(y_test, test_predictions))

print("Final Test Results")
print("R²:", round(test_r2, 4))
print("MAE: £", round(test_mae, 2))
print("RMSE: £", round(test_rmse, 2))

## 24. Actual vs Predicted Price

In [ ]:
plt.figure(figsize=(7, 7))
plt.scatter(y_test, test_predictions, alpha=0.3)

minimum = min(y_test.min(), test_predictions.min())
maximum = max(y_test.max(), test_predictions.max())

plt.plot([minimum, maximum], [minimum, maximum], "r--")

plt.xlabel("Actual Price (£)")
plt.ylabel("Predicted Price (£)")
plt.title("Actual vs Predicted Price")
plt.show()

## 25. Prediction Error Analysis

In [ ]:
errors = X_test.copy()

errors["ActualPrice"] = y_test.values
errors["PredictedPrice"] = test_predictions
errors["Error"] = errors["PredictedPrice"] - errors["ActualPrice"]
errors["AbsoluteError"] = abs(errors["Error"])

largest_errors = errors.sort_values(
    "AbsoluteError",
    ascending=False
).head(15)

largest_errors[
    [
        "Make",
        "model",
        "year",
        "mileage",
        "transmission",
        "fuelType",
        "ActualPrice",
        "PredictedPrice",
        "AbsoluteError"
    ]
]

Look at the cars with the largest errors and check whether they are rare models, unusually expensive cars, very high-mileage cars, or examples that are not common in the training data.

## 26. Feature Importance

In [ ]:
if final_model_name == "Tuned Random Forest":
    fitted_preprocessor = final_model.named_steps["preprocessing"]
    fitted_rf = final_model.named_steps["model"]

    feature_names = fitted_preprocessor.get_feature_names_out()
    importance_values = fitted_rf.feature_importances_

else:
    feature_names = preprocessor.get_feature_names_out()
    importance_values = random_forest.feature_importances_

feature_importance = pd.DataFrame({
    "Feature": feature_names,
    "Importance": importance_values
})

feature_importance["Feature"] = (
    feature_importance["Feature"]
    .str.replace("num__", "", regex=False)
    .str.replace("cat__", "", regex=False)
)

feature_importance = feature_importance.sort_values(
    "Importance",
    ascending=False
).head(20)

feature_importance

In [ ]:
plt.figure(figsize=(10, 7))
sns.barplot(
    data=feature_importance,
    x="Importance",
    y="Feature"
)
plt.title("Top 20 Most Important Features")
plt.show()

## 27. Smart Deal Advisor

The seller price is used **after** the model predicts the market price.

It is not included in model training.

In [ ]:
car = pd.DataFrame({
    "model": ["A4"],
    "year": [2018],
    "transmission": ["Automatic"],
    "mileage": [25000],
    "fuelType": ["Diesel"],
    "tax": [145],
    "mpg": [55.0],
    "engineSize": [2.0],
    "Make": ["Audi"]
})

seller_price = 17000

In [ ]:
car["CarAge"] = 2021 - car["year"]
car["MileagePerYear"] = car["mileage"] / car["CarAge"].replace(0, 1)
car["CarAgeSquared"] = car["CarAge"] ** 2

car

In [ ]:
if final_model_name == "Tuned Random Forest":
    predicted_price = final_model.predict(car)[0]
else:
    car_processed = preprocessor.transform(car)
    predicted_price = random_forest.predict(car_processed)[0]

difference = seller_price - predicted_price
percentage_difference = (difference / predicted_price) * 100

print("Estimated Market Price: £", round(predicted_price, 2))
print("Seller Price: £", seller_price)
print("Difference: £", round(abs(difference), 2))
print("Percentage Difference:", round(percentage_difference, 2), "%")

In [ ]:
if percentage_difference < -10:
    deal_rating = "🟢 GREAT DEAL"

elif percentage_difference < -5:
    deal_rating = "🟢 GOOD DEAL"

elif percentage_difference <= 5:
    deal_rating = "🟡 FAIR PRICE"

elif percentage_difference <= 10:
    deal_rating = "🟠 SLIGHTLY OVERPRICED"

else:
    deal_rating = "🔴 OVERPRICED"

print("Deal Rating:", deal_rating)

## 28. Save the Final Model

Saving the whole tuned pipeline is easiest because it stores both preprocessing and the Random Forest model together.

In [ ]:
if final_model_name == "Tuned Random Forest":
    joblib.dump(final_model, "autoworth_model.pkl")
    print("Tuned model saved.")

else:
    joblib.dump(preprocessor, "autoworth_preprocessor.pkl")
    joblib.dump(random_forest, "autoworth_model.pkl")
    print("Untuned model and preprocessor saved.")

# Final Conclusion

After running all cells, the final conclusion should include:

- final number of records after cleaning
- important data-quality findings
- the main EDA observations
- the 3 engineered features
- Linear Regression baseline results
- comparison of the 4 regression models
- training vs validation performance
- whether tuning improved Random Forest
- final test R², MAE and RMSE
- important features
- examples of large prediction errors
- how the Smart Deal Advisor works

The main goal is not only to get a high R². The workflow should be correct and the final test results should represent performance on unseen data.